# 🎯 OpenCV in Python — Comprehensive Learning Notebook

**Audience:** Experienced Python developer (8–9 years), familiar with CV concepts and PyTorch CNN pipelines.  
**Goal:** Go from zero OpenCV to production-ready proficiency — covering the full image processing pipeline that feeds real CV/ML systems.

---

## 📚 Table of Contents

1. [Setup & Core Concepts](#1-setup)
2. [Image I/O, Color Spaces & Channels](#2-io)
3. [Drawing, Annotations & ROI](#3-drawing)
4. [Geometric Transformations](#4-geometric)
5. [Image Filtering & Enhancement](#5-filtering)
6. [Thresholding & Morphological Operations](#6-morphology)
7. [Edge & Gradient Detection](#7-edges)
8. [Contours, Shapes & Moments](#8-contours)
9. [Histograms & Color Analysis](#9-histograms)
10. [Feature Detection & Matching (SIFT, ORB)](#10-features)
11. [Object Detection — Haar Cascades & HOG](#11-detection)
12. [Optical Flow & Background Subtraction](#12-video)
13. [Camera Calibration & Homography](#13-calibration)
14. [Integration with NumPy & PyTorch Tensors](#14-pytorch)
15. [Real-World Pipeline: Preprocessing for CNN](#15-pipeline)

---
> **Tip:** Each section is self-contained. Run cells top-to-bottom within a section. Inline comments explain the *why*, not just the *what*.


## 1. Setup & Core Concepts <a id='1-setup'></a>

In [ ]:
# Standard imports — you'll use these in every OpenCV script
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

print(f"OpenCV version : {cv2.__version__}")
print(f"NumPy version  : {np.__version__}")

# ── Helper: inline display (replaces cv2.imshow in notebooks) ──────────────
def show(img, title="", cmap=None, figsize=(8, 5)):
    """Display BGR, grayscale, or float image inline."""
    plt.figure(figsize=figsize)
    if img.ndim == 2 or cmap is not None:          # grayscale
        plt.imshow(img, cmap=cmap or "gray")
    else:                                           # BGR → RGB conversion
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title); plt.axis("off"); plt.tight_layout(); plt.show()

def show_multi(*args, cols=3, figsize=(14, 5)):
    """Display multiple (title, img) tuples side-by-side."""
    rows = (len(args) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).flatten()
    for ax, (title, img) in zip(axes, args):
        if img.ndim == 2:
            ax.imshow(img, cmap="gray")
        else:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(title); ax.axis("off")
    for ax in axes[len(args):]: ax.axis("off")   # hide unused subplots
    plt.tight_layout(); plt.show()

print("Helpers loaded ✓")


### 🔑 Key Mental Model: OpenCV ↔ NumPy

| Concept | OpenCV | NumPy equivalent |
|---------|--------|-----------------|
| Image | `np.ndarray` (H, W, C) dtype=uint8 | same array |
| Color order | **BGR** (not RGB!) | axis=-1 channels |
| Origin | top-left (y down) | row 0 = top |
| Pixel access | `img[y, x]` | `arr[row, col]` |
| Copy | `img.copy()` | `arr.copy()` |

> **Critical gotcha:** OpenCV uses BGR everywhere. Matplotlib expects RGB. Always convert before displaying — the `show()` helper above handles this.


In [ ]:
# ── Synthetic test image (no file needed) ─────────────────────────────────
# Create a 300×400 BGR image programmatically
canvas = np.zeros((300, 400, 3), dtype=np.uint8)

# Paint quadrants to visualise channel order
canvas[:150, :200] = (255, 0, 0)    # Blue (BGR!)
canvas[:150, 200:] = (0, 255, 0)    # Green
canvas[150:, :200] = (0, 0, 255)    # Red
canvas[150:, 200:] = (128, 128, 0)  # Teal

show(canvas, "Synthetic canvas — quadrant colours")

# ── Image metadata you'll query constantly ─────────────────────────────────
h, w, c = canvas.shape
print(f"Shape  : {canvas.shape}   → H={h}, W={w}, Channels={c}")
print(f"dtype  : {canvas.dtype}")
print(f"Range  : [{canvas.min()}, {canvas.max()}]")
print(f"Memory : {canvas.nbytes / 1024:.1f} KB")

# ── Pixel access ───────────────────────────────────────────────────────────
px = canvas[10, 10]          # BGR tuple at (row=10, col=10)
print(f"Pixel at (10,10) BGR: {px}")


---
## 2. Image I/O, Color Spaces & Channels <a id='2-io'></a>

In [ ]:
# ── Create a realistic test image for the rest of the notebook ────────────
def make_test_image(h=480, w=640):
    """Gradient + geometric shapes — portable, no file dependency."""
    img = np.zeros((h, w, 3), dtype=np.uint8)
    # Gradient background
    for i in range(h):
        img[i, :, 0] = int(255 * i / h)       # B channel gradient
    for j in range(w):
        img[:, j, 2] = int(255 * j / w)       # R channel gradient
    # Shapes
    cv2.circle(img, (320, 240), 100, (0, 255, 0), -1)       # filled green circle
    cv2.rectangle(img, (50, 50), (200, 150), (255, 255, 0), 3)
    cv2.ellipse(img, (500, 380), (80, 50), 30, 0, 360, (0, 165, 255), -1)
    cv2.putText(img, "OpenCV Test", (160, 440),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 2)
    return img

img = make_test_image()
show(img, "Test image (used throughout the notebook)")

# ── Encode / decode in memory (useful for web pipelines) ──────────────────
_, buf = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 90])
img_decoded = cv2.imdecode(buf, cv2.IMREAD_COLOR)
print(f"Round-trip JPEG: original {img.shape}, decoded {img_decoded.shape}")

# ── Save to disk ────────────────────────────────────────────────────────────
cv2.imwrite("/tmp/test_img.png", img)
print("Saved to /tmp/test_img.png")

# ── Load from disk ─────────────────────────────────────────────────────────
loaded = cv2.imread("/tmp/test_img.png", cv2.IMREAD_COLOR)   # BGR uint8
gray   = cv2.imread("/tmp/test_img.png", cv2.IMREAD_GRAYSCALE)
print(f"Loaded colour: {loaded.shape}, grayscale: {gray.shape}")


In [ ]:
# ── Color space conversions ────────────────────────────────────────────────
# OpenCV supports 200+ cvtColor codes. Most important ones:
bgr   = img.copy()
rgb   = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
gray  = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
hsv   = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
lab   = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
yuv   = cv2.cvtColor(bgr, cv2.COLOR_BGR2YUV)

show_multi(
    ("BGR (native)", bgr),
    ("Grayscale",    gray),
    ("HSV",          hsv),
    ("LAB",          lab),
    cols=4,
)

# ── Why HSV? → colour-based segmentation ──────────────────────────────────
# HSV separates Hue from luminance → robust to lighting changes.
# H ∈ [0,179]  S ∈ [0,255]  V ∈ [0,255]   (OpenCV convention)
lower_green = np.array([40,  50,  50])
upper_green = np.array([80, 255, 255])
mask = cv2.inRange(hsv, lower_green, upper_green)

result = cv2.bitwise_and(bgr, bgr, mask=mask)
show_multi(("Original", bgr), ("HSV green mask", mask), ("Isolated green", result))


In [ ]:
# ── Channel splitting & merging ────────────────────────────────────────────
b, g, r = cv2.split(img)           # returns three 2-D arrays
merged   = cv2.merge([b, g, r])    # reconstruct

# Visualise individual channels as grayscale
show_multi(
    ("Blue channel",  b),
    ("Green channel", g),
    ("Red channel",   r),
)

# ── Direct NumPy slicing (faster than cv2.split for single channel) ────────
r_fast = img[:, :, 2]              # no copy — shared memory view
b_copy = img[:, :, 0].copy()       # explicit copy when you'll modify

# ── Add alpha channel (BGRA) ───────────────────────────────────────────────
alpha = np.full((img.shape[0], img.shape[1]), 200, dtype=np.uint8)  # 200/255 opacity
bgra  = cv2.merge([b, g, r, alpha])
print(f"BGRA image shape: {bgra.shape}")


---
## 3. Drawing, Annotations & ROI <a id='3-drawing'></a>

In [ ]:
# All draw functions operate IN-PLACE and return None.
# Always work on a copy if you want to preserve the original.

canvas = np.ones((500, 700, 3), dtype=np.uint8) * 30  # dark background

# ── Primitives ─────────────────────────────────────────────────────────────
cv2.line(canvas, (50, 50), (650, 50), (0, 255, 255), 2)
cv2.rectangle(canvas, (50, 80), (250, 200), (255, 100, 0), 3)
cv2.rectangle(canvas, (280, 80), (480, 200), (0, 200, 255), -1)  # -1 = filled

cv2.circle(canvas, (580, 140), 60, (0, 255, 0), 4)
cv2.circle(canvas, (580, 140), 10, (255, 255, 255), -1)  # centre dot

cv2.ellipse(canvas, (150, 320), (100, 60), 0, 0, 360, (200, 50, 255), 3)
cv2.ellipse(canvas, (150, 320), (100, 60), 45, 0, 180, (255, 200, 0), -1)  # arc

# ── Polylines & polygons ────────────────────────────────────────────────────
pts = np.array([[350,250],[450,300],[430,400],[270,400],[250,300]], np.int32)
pts = pts.reshape((-1, 1, 2))             # required shape for polylines
cv2.polylines(canvas, [pts], isClosed=True, color=(0, 200, 100), thickness=2)
cv2.fillPoly(canvas, [pts + np.array([200, 0])], color=(0, 100, 200))

# ── Text ────────────────────────────────────────────────────────────────────
fonts = [cv2.FONT_HERSHEY_SIMPLEX, cv2.FONT_HERSHEY_DUPLEX,
         cv2.FONT_HERSHEY_TRIPLEX, cv2.FONT_ITALIC]
for i, font in enumerate(fonts):
    cv2.putText(canvas, f"Font {i}", (50 + i*160, 460), font, 0.7, (220,220,220), 1)

# ── Get text bounding box (useful for label backgrounds) ──────────────────
label = "Detection: 0.97"
(tw, th), baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.8, 2)
x, y = 50, 150
cv2.rectangle(canvas, (x, y - th - baseline - 4), (x + tw + 4, y + baseline), (0,0,0), -1)
cv2.putText(canvas, label, (x+2, y-2), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,100), 2)

show(canvas, "Drawing primitives", figsize=(12, 7))


In [ ]:
# ── Region of Interest (ROI) ───────────────────────────────────────────────
# ROI is just NumPy slicing: img[y:y+h, x:x+w]
# This is a VIEW (shared memory) — modify with care.

src = make_test_image()

# Extract ROI
roi = src[100:300, 200:450].copy()       # copy for independent manipulation
show(roi, "Extracted ROI")

# Paste a modified ROI back
roi_blur = cv2.GaussianBlur(roi, (21, 21), 0)
dst = src.copy()
dst[100:300, 200:450] = roi_blur

show_multi(("Original", src), ("ROI blurred in-place", dst))

# ── Masking with arbitrary shapes ──────────────────────────────────────────
mask_roi = np.zeros(src.shape[:2], dtype=np.uint8)
cv2.circle(mask_roi, (320, 240), 120, 255, -1)   # circular mask
result = cv2.bitwise_and(src, src, mask=mask_roi)
show_multi(("Original", src), ("Circular mask", mask_roi), ("Masked output", result))


---
## 4. Geometric Transformations <a id='4-geometric'></a>

In [ ]:
src = make_test_image()
h, w = src.shape[:2]

# ── Resize ─────────────────────────────────────────────────────────────────
# INTER_AREA  → shrink (anti-aliased, preferred for downscale)
# INTER_CUBIC / INTER_LANCZOS4 → enlarge (better quality, slower)
# INTER_LINEAR → enlarge (default, good balance)
small  = cv2.resize(src, (320, 240), interpolation=cv2.INTER_AREA)
large  = cv2.resize(src, (960, 720), interpolation=cv2.INTER_CUBIC)
# Resize by scale factor
half   = cv2.resize(src, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA)

print(f"Original: {src.shape}  Small: {small.shape}  Large: {large.shape}")

# ── Flip ───────────────────────────────────────────────────────────────────
flip_h = cv2.flip(src, 1)   # horizontal
flip_v = cv2.flip(src, 0)   # vertical
flip_b = cv2.flip(src, -1)  # both

show_multi(
    ("Original", src), ("Flip H", flip_h), ("Flip V", flip_v), ("Flip Both", flip_b)
)


In [ ]:
src = make_test_image()
h, w = src.shape[:2]

# ── Rotation (warpAffine route) ────────────────────────────────────────────
cx, cy = w // 2, h // 2
M_rot = cv2.getRotationMatrix2D((cx, cy), angle=30, scale=1.0)
rotated = cv2.warpAffine(src, M_rot, (w, h), flags=cv2.INTER_LINEAR,
                         borderMode=cv2.BORDER_REFLECT_101)

# ── Translation ────────────────────────────────────────────────────────────
tx, ty = 80, 50
M_trans = np.float32([[1, 0, tx], [0, 1, ty]])
translated = cv2.warpAffine(src, M_trans, (w, h))

# ── Affine transform (3-point) ─────────────────────────────────────────────
pts1 = np.float32([[50,50],[200,50],[50,200]])
pts2 = np.float32([[10,80],[220,30],[80,250]])
M_aff = cv2.getAffineTransform(pts1, pts2)
affine = cv2.warpAffine(src, M_aff, (w, h))

# ── Perspective transform (homography, 4-point) ────────────────────────────
pts1 = np.float32([[100,100],[540,100],[100,380],[540,380]])
pts2 = np.float32([[0,0],[w,0],[0,h],[w,h]])
M_per = cv2.getPerspectiveTransform(pts1, pts2)
perspective = cv2.warpPerspective(src, M_per, (w, h))

show_multi(
    ("Original",     src),
    ("Rotated 30°",  rotated),
    ("Translated",   translated),
    ("Affine",       affine),
    ("Perspective",  perspective),
    cols=3,
)


In [ ]:
# ── Cropping + padding for CNN preprocessing ──────────────────────────────
# Common pattern: letterbox resize (preserve aspect ratio, pad to square)

def letterbox(img, target=640, fill=114):
    """Resize keeping aspect ratio, pad to (target × target)."""
    h, w = img.shape[:2]
    scale = target / max(h, w)
    nh, nw = int(h * scale), int(w * scale)
    resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
    out = np.full((target, target, 3), fill, dtype=np.uint8)
    y0, x0 = (target - nh) // 2, (target - nw) // 2
    out[y0:y0+nh, x0:x0+nw] = resized
    return out, scale, (x0, y0)

lb, scale, (px, py) = letterbox(src, 640)
show_multi(("Original (480×640)", src), (f"Letterboxed (640×640)", lb))
print(f"Scale: {scale:.3f}, Padding offset: ({px}, {py})")


---
## 5. Image Filtering & Enhancement <a id='5-filtering'></a>

In [ ]:
src = make_test_image()

# ── Add noise for demo purposes ────────────────────────────────────────────
noise = np.random.normal(0, 30, src.shape).astype(np.int16)
noisy = np.clip(src.astype(np.int16) + noise, 0, 255).astype(np.uint8)

# ── Smoothing filters ───────────────────────────────────────────────────────
# Box blur (mean) — fast, not great
box    = cv2.blur(noisy, (7, 7))

# Gaussian — weighted mean, best for noise removal without destroying edges too much
gauss  = cv2.GaussianBlur(noisy, (7, 7), sigmaX=0)  # sigmaX=0 → computed from ksize

# Median — excellent for salt-and-pepper noise, preserves edges
median = cv2.medianBlur(noisy, 7)

# Bilateral — edge-preserving smoothing (expensive but high quality)
bilat  = cv2.bilateralFilter(noisy, d=9, sigmaColor=75, sigmaSpace=75)

show_multi(
    ("Noisy input",   noisy),
    ("Box blur",      box),
    ("Gaussian blur", gauss),
    ("Median blur",   median),
    ("Bilateral",     bilat),
    cols=3,
)


In [ ]:
src = make_test_image()

# ── Custom kernels with filter2D ───────────────────────────────────────────
# Sharpening kernel
K_sharp = np.array([[ 0,-1, 0],
                    [-1, 5,-1],
                    [ 0,-1, 0]], dtype=np.float32)
sharp = cv2.filter2D(src, -1, K_sharp)   # ddepth=-1 → same as src

# Emboss
K_emboss = np.array([[-2,-1, 0],
                     [-1, 1, 1],
                     [ 0, 1, 2]], dtype=np.float32)
emboss = cv2.filter2D(cv2.cvtColor(src, cv2.COLOR_BGR2GRAY), -1, K_emboss)

# Edge detection kernel (Laplacian-like)
K_edge = np.array([[-1,-1,-1],
                   [-1, 8,-1],
                   [-1,-1,-1]], dtype=np.float32)
edges_k = cv2.filter2D(src, -1, K_edge)

show_multi(("Original", src), ("Sharpen", sharp), ("Emboss", emboss), ("Edge kernel", edges_k))

# ── Morphological operations (preview — detailed in Section 6) ─────────────
# These also use filter2D internally with structuring elements


In [ ]:
# ── Contrast & brightness adjustments ─────────────────────────────────────
src = make_test_image()

# Method 1: linear α·x + β
bright = cv2.convertScaleAbs(src, alpha=1.3, beta=40)
dark   = cv2.convertScaleAbs(src, alpha=0.7, beta=-30)

# Method 2: CLAHE (Contrast Limited Adaptive Histogram Equalization)
# Much better than global equalisation for natural images
gray = cv2.cvtColor(src, cv2.COLOR_BGR2GRAY)
clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
eq    = clahe.apply(gray)

# Method 3: Gamma correction
gamma = 2.0
lut = np.array([((i / 255.0) ** (1.0 / gamma)) * 255
                for i in np.arange(256)], dtype=np.uint8)
gamma_img = cv2.LUT(src, lut)

show_multi(
    ("Original",     src),
    ("Bright/contrast", bright),
    ("Dark",         dark),
    ("CLAHE",        eq),
    ("Gamma 2.0",    gamma_img),
    cols=3,
)


---
## 6. Thresholding & Morphological Operations <a id='6-morphology'></a>

In [ ]:
src = make_test_image()
gray = cv2.cvtColor(src, cv2.COLOR_BGR2GRAY)

# ── Thresholding methods ───────────────────────────────────────────────────
_, thresh_bin   = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
_, thresh_otsu  = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
thresh_adapt    = cv2.adaptiveThreshold(
    gray, 255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,    # or ADAPTIVE_THRESH_MEAN_C
    cv2.THRESH_BINARY,
    blockSize=31,  # neighbourhood size (odd number)
    C=5,           # constant subtracted from mean
)

print(f"Otsu's optimal threshold: {thresh_otsu[0]:.1f}")  # first return is threshold

show_multi(
    ("Grayscale",           gray),
    ("Binary @127",         thresh_bin),
    ("Otsu auto-threshold", thresh_otsu[1]),
    ("Adaptive Gaussian",   thresh_adapt),
)


In [ ]:
# ── Morphological operations ───────────────────────────────────────────────
# All ops work on BINARY images (0 or 255).
gray = cv2.cvtColor(make_test_image(), cv2.COLOR_BGR2GRAY)
_, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# Structuring element (kernel)
kernel_3 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
kernel_l = cv2.getStructuringElement(cv2.MORPH_RECT,    (15, 15))

erosion  = cv2.erode(binary, kernel_3, iterations=1)   # shrinks white regions
dilation = cv2.dilate(binary, kernel_3, iterations=1)  # expands white regions
opening  = cv2.morphologyEx(binary, cv2.MORPH_OPEN,  kernel_3)  # erosion → dilation: remove noise
closing  = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel_3)  # dilation → erosion: fill holes
gradient = cv2.morphologyEx(binary, cv2.MORPH_GRADIENT, kernel_3)  # dilation - erosion: edge outline
tophat   = cv2.morphologyEx(binary, cv2.MORPH_TOPHAT,   kernel_l)  # src - opening: bright details
blackhat = cv2.morphologyEx(binary, cv2.MORPH_BLACKHAT, kernel_l)  # closing - src: dark details

show_multi(
    ("Binary",    binary),
    ("Erosion",   erosion),
    ("Dilation",  dilation),
    ("Opening",   opening),
    ("Closing",   closing),
    ("Gradient",  gradient),
    cols=3,
)

print("""
Practical guide:
  Opening  → remove small white noise blobs on dark background
  Closing  → fill small holes/gaps in white regions
  Gradient → extract object borders
  Top-hat  → detect bright regions smaller than the kernel
  Black-hat→ detect dark regions smaller than the kernel
""")


---
## 7. Edge & Gradient Detection <a id='7-edges'></a>

In [ ]:
gray = cv2.cvtColor(make_test_image(), cv2.COLOR_BGR2GRAY)
blurred = cv2.GaussianBlur(gray, (3, 3), 0)   # always blur before edge detection

# ── Sobel (directional gradient) ───────────────────────────────────────────
sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)   # horizontal gradient
sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)   # vertical gradient
sobel_mag = cv2.magnitude(sobel_x, sobel_y)
sobel_mag = cv2.normalize(sobel_mag, None, 0, 255, cv2.NORM_MINMAX, cv2.CV_8U)

# Gradient direction (radians)
sobel_angle = np.arctan2(np.abs(sobel_y), np.abs(sobel_x))

# ── Laplacian (second-order, isotropic) ───────────────────────────────────
lap = cv2.Laplacian(blurred, cv2.CV_64F)
lap_abs = cv2.convertScaleAbs(lap)

# ── Canny (multi-stage, gold standard for edges) ──────────────────────────
# auto-threshold trick: use median pixel value to set thresholds
sigma = 0.33
v = float(np.median(blurred))
lo, hi = max(0, (1.0 - sigma) * v), min(255, (1.0 + sigma) * v)
canny = cv2.Canny(blurred, lo, hi)

print(f"Auto Canny thresholds: lo={lo:.0f}, hi={hi:.0f}")

show_multi(
    ("Gray",          gray),
    ("Sobel X",       cv2.convertScaleAbs(sobel_x)),
    ("Sobel Y",       cv2.convertScaleAbs(sobel_y)),
    ("Sobel Mag",     sobel_mag),
    ("Laplacian",     lap_abs),
    ("Canny",         canny),
    cols=3,
)


In [ ]:
# ── Hough Line Transform ───────────────────────────────────────────────────
src = make_test_image()
edges = cv2.Canny(cv2.cvtColor(src, cv2.COLOR_BGR2GRAY), 50, 150)

# Probabilistic Hough (faster, returns line segments)
lines = cv2.HoughLinesP(edges, rho=1, theta=np.pi/180,
                         threshold=80, minLineLength=60, maxLineGap=10)

canvas = src.copy()
if lines is not None:
    for x1, y1, x2, y2 in lines[:, 0]:
        cv2.line(canvas, (x1, y1), (x2, y2), (0, 255, 0), 2)
    print(f"Found {len(lines)} line segments")

# ── Hough Circle Transform ──────────────────────────────────────────────────
gray_b = cv2.GaussianBlur(cv2.cvtColor(src, cv2.COLOR_BGR2GRAY), (9, 9), 2)
circles = cv2.HoughCircles(
    gray_b, cv2.HOUGH_GRADIENT, dp=1.2,
    minDist=50, param1=100, param2=30, minRadius=30, maxRadius=120
)

if circles is not None:
    circles = np.round(circles[0]).astype(int)
    for x, y, r in circles:
        cv2.circle(canvas, (x, y), r, (0, 165, 255), 2)
        cv2.circle(canvas, (x, y), 3,  (255, 0, 255), -1)
    print(f"Found {len(circles)} circles")

show_multi(("Edges (Canny)", edges), ("Lines + Circles", canvas))


---
## 8. Contours, Shapes & Moments <a id='8-contours'></a>

In [ ]:
src = make_test_image()
gray = cv2.cvtColor(src, cv2.COLOR_BGR2GRAY)
_, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# ── Find contours ──────────────────────────────────────────────────────────
# RETR_EXTERNAL → outer contours only
# RETR_TREE      → full hierarchy
# CHAIN_APPROX_SIMPLE → compress horizontal/vertical/diagonal segments
contours, hierarchy = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
print(f"Found {len(contours)} contours")

canvas = src.copy()
# Draw all contours
cv2.drawContours(canvas, contours, -1, (0, 255, 0), 2)

# ── Contour properties ─────────────────────────────────────────────────────
for i, cnt in enumerate(sorted(contours, key=cv2.contourArea, reverse=True)[:5]):
    area      = cv2.contourArea(cnt)
    perimeter = cv2.arcLength(cnt, closed=True)
    
    # Bounding geometries
    x, y, w, h = cv2.boundingRect(cnt)               # axis-aligned bbox
    rect = cv2.minAreaRect(cnt)                        # rotated bbox
    box  = cv2.boxPoints(rect).astype(int)
    (cx, cy), radius = cv2.minEnclosingCircle(cnt)    # min circle
    
    # Circularity (1.0 = perfect circle)
    if perimeter > 0:
        circularity = 4 * np.pi * area / (perimeter ** 2)
    
    # Convex hull
    hull = cv2.convexHull(cnt)
    
    # Moments → centroid
    M = cv2.moments(cnt)
    if M["m00"] != 0:
        mu_x = int(M["m10"] / M["m00"])
        mu_y = int(M["m01"] / M["m00"])
        cv2.circle(canvas, (mu_x, mu_y), 5, (255, 0, 0), -1)
    
    cv2.polylines(canvas, [box], True, (255, 165, 0), 2)  # rotated bbox
    cv2.circle(canvas, (int(cx), int(cy)), int(radius), (0, 0, 255), 1)
    
    print(f"Contour {i}: area={area:.0f}, circ={circularity:.2f}, "
          f"bbox=({x},{y},{w},{h})")

show(canvas, "Contours with bounding geometries & centroids", figsize=(10, 7))


In [ ]:
# ── Shape approximation (Douglas-Peucker) ─────────────────────────────────
src = make_test_image()
gray = cv2.cvtColor(src, cv2.COLOR_BGR2GRAY)
_, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

canvas = src.copy()
for cnt in contours:
    if cv2.contourArea(cnt) < 500:
        continue
    epsilon = 0.02 * cv2.arcLength(cnt, True)   # 2% of perimeter
    approx = cv2.approxPolyDP(cnt, epsilon, True)
    
    n = len(approx)
    color = {3: (0,255,0), 4: (255,0,0), 5: (0,165,255)}.get(n, (128,0,128))
    cv2.drawContours(canvas, [approx], 0, color, 3)
    
    # classify
    if n == 3:   label = "Triangle"
    elif n == 4: label = "Quad"
    elif n == 5: label = "Pentagon"
    else:        label = f"Poly({n})"
    
    M = cv2.moments(cnt)
    if M["m00"]:
        cx, cy = int(M["m10"]/M["m00"]), int(M["m01"]/M["m00"])
        cv2.putText(canvas, label, (cx-30, cy), cv2.FONT_HERSHEY_SIMPLEX,
                    0.5, (255,255,255), 1)

show(canvas, "Shape classification via polygon approximation")


---
## 9. Histograms & Color Analysis <a id='9-histograms'></a>

In [ ]:
src = make_test_image()

# ── Compute histograms ─────────────────────────────────────────────────────
colors = {"B": (255, 50, 50), "G": (50, 200, 50), "R": (50, 50, 255)}
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Per-channel histogram
ax = axes[0]
for i, (ch_name, color) in enumerate(colors.items()):
    hist = cv2.calcHist([src], [i], None, [256], [0, 256])
    ax.plot(hist, color=[c/255 for c in color], linewidth=1.5, label=ch_name)
ax.set_title("Per-channel histogram"); ax.legend(); ax.set_xlim([0, 256])
ax.set_xlabel("Pixel intensity"); ax.set_ylabel("Count")

# Grayscale histogram
gray = cv2.cvtColor(src, cv2.COLOR_BGR2GRAY)
hist_gray = cv2.calcHist([gray], [0], None, [256], [0, 256])
axes[1].plot(hist_gray, color="gray", linewidth=1.5)
axes[1].set_title("Grayscale histogram"); axes[1].set_xlim([0, 256])
axes[1].set_xlabel("Pixel intensity"); axes[1].set_ylabel("Count")
plt.tight_layout(); plt.show()

# ── Histogram equalisation ─────────────────────────────────────────────────
eq_gray = cv2.equalizeHist(gray)
show_multi(("Original gray", gray), ("Equalised", eq_gray))


In [ ]:
# ── Histogram back-projection: track an object by color ──────────────────
# 1. Build a histogram of the "target" color sample
# 2. Back-project it on the full image → each pixel gets a probability score

src = make_test_image()
hsv = cv2.cvtColor(src, cv2.COLOR_BGR2HSV)

# Sample: the green circle region (rough ROI)
roi_sample = hsv[140:340, 220:420]
roi_hist   = cv2.calcHist([roi_sample], [0, 1], None, [180, 256], [0, 180, 0, 256])
cv2.normalize(roi_hist, roi_hist, 0, 255, cv2.NORM_MINMAX)

# Back-project
back_proj = cv2.calcBackProject([hsv], [0, 1], roi_hist, [0, 180, 0, 256], scale=1)

# Clean up with morphology
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
back_proj = cv2.filter2D(back_proj, -1, kernel)

show_multi(
    ("Original", src),
    ("Backprojection (green probability)", back_proj),
)


---
## 10. Feature Detection & Matching (SIFT, ORB) <a id='10-features'></a>

In [ ]:
src = make_test_image()

# ── Harris Corner Detection ────────────────────────────────────────────────
gray = cv2.cvtColor(src, cv2.COLOR_BGR2GRAY).astype(np.float32)
harris = cv2.cornerHarris(gray, blockSize=2, ksize=3, k=0.04)
harris = cv2.dilate(harris, None)   # dilate to mark corners bigger

canvas_h = src.copy()
canvas_h[harris > 0.01 * harris.max()] = [0, 0, 255]  # paint corners red
show(canvas_h, "Harris corners")

# ── Shi-Tomasi (Good Features To Track) ────────────────────────────────────
gray_u8 = cv2.cvtColor(src, cv2.COLOR_BGR2GRAY)
corners = cv2.goodFeaturesToTrack(gray_u8, maxCorners=50, qualityLevel=0.01, minDistance=10)

canvas_st = src.copy()
if corners is not None:
    for corner in corners.astype(int):
        x, y = corner.ravel()
        cv2.circle(canvas_st, (x, y), 5, (0, 255, 0), -1)
show(canvas_st, f"Shi-Tomasi: {len(corners)} corners")


In [ ]:
src = make_test_image()
gray = cv2.cvtColor(src, cv2.COLOR_BGR2GRAY)

# ── ORB (free, fast — good for real-time) ─────────────────────────────────
orb = cv2.ORB_create(nfeatures=500)
kp_orb, desc_orb = orb.detectAndCompute(gray, None)
print(f"ORB: {len(kp_orb)} keypoints, descriptor shape: {desc_orb.shape}")

canvas_orb = cv2.drawKeypoints(
    src, kp_orb, None,
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS,
    color=(0, 255, 0),
)

# ── SIFT (patented until 2020, now free in OpenCV 4.4+) ───────────────────
sift = cv2.SIFT_create(nfeatures=300)
kp_sift, desc_sift = sift.detectAndCompute(gray, None)
print(f"SIFT: {len(kp_sift)} keypoints, descriptor shape: {desc_sift.shape}")

canvas_sift = cv2.drawKeypoints(
    src, kp_sift, None,
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS,
    color=(0, 165, 255),
)

show_multi(("ORB keypoints", canvas_orb), ("SIFT keypoints", canvas_sift))


In [ ]:
# ── Feature matching: ORB + BFMatcher ─────────────────────────────────────
src = make_test_image()
gray = cv2.cvtColor(src, cv2.COLOR_BGR2GRAY)

# Create a second "query" image — rotated/scaled version of original
M = cv2.getRotationMatrix2D((320, 240), 20, 0.8)
query = cv2.warpAffine(gray, M, (640, 480))

orb = cv2.ORB_create(1000)
kp1, d1 = orb.detectAndCompute(gray, None)
kp2, d2 = orb.detectAndCompute(query, None)

# BFMatcher (brute-force) with Hamming distance for binary descriptors
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
matches = bf.knnMatch(d1, d2, k=2)

# Lowe's ratio test — filter out ambiguous matches
good = [m for m, n in matches if m.distance < 0.75 * n.distance]
print(f"Total matches: {len(matches)} → After ratio test: {len(good)}")

match_img = cv2.drawMatchesKnn(
    src, kp1, query, kp2,
    [[m] for m in good[:40]], None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
)
show(match_img, f"ORB feature matching ({len(good)} good matches)", figsize=(14, 5))


In [ ]:
# ── FLANN matcher (faster for large descriptor sets) ─────────────────────
src = make_test_image()
gray = cv2.cvtColor(src, cv2.COLOR_BGR2GRAY)

# SIFT with FLANN
sift = cv2.SIFT_create(500)
kp1, d1 = sift.detectAndCompute(gray, None)

query_gray = cv2.warpAffine(gray,
    cv2.getRotationMatrix2D((320, 240), 15, 0.85), (640, 480))
kp2, d2 = sift.detectAndCompute(query_gray, None)

FLANN_INDEX_KDTREE = 1
index_params  = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
search_params = dict(checks=50)
flann = cv2.FlannBasedMatcher(index_params, search_params)

matches = flann.knnMatch(d1, d2, k=2)
good = [m for m, n in matches if m.distance < 0.7 * n.distance]
print(f"FLANN good matches: {len(good)}")

# ── Homography from matches ────────────────────────────────────────────────
if len(good) >= 4:
    src_pts = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
    H, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)
    inliers = mask.ravel().sum()
    print(f"Homography found, {inliers}/{len(good)} inliers")
    print(f"H matrix:\n{H}")


---
## 11. Object Detection — Haar Cascades & HOG + SVM <a id='11-detection'></a>

In [ ]:
# ── HOG (Histogram of Oriented Gradients) person detector ─────────────────
# Built-in OpenCV — no pre-trained model file needed

hog = cv2.HOGDescriptor()
hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())

# Create a synthetic "test frame" (in practice: load video frame)
test_frame = np.ones((480, 640, 3), dtype=np.uint8) * 180
cv2.putText(test_frame, "HOG person detector ready", (80, 240),
            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 180), 2)

# Detect (on a real photo with people)
# rects, weights = hog.detectMultiScale(
#     frame,
#     winStride=(8, 8),
#     padding=(8, 8),
#     scale=1.05,
#     useMeanshiftGrouping=False,  # NMS alternative
# )
# for (x, y, w, h) in rects:
#     cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)

show(test_frame, "HOG detector initialised — connect a real frame to run detection")
print("HOG descriptor size:", hog.getDescriptorSize())


In [ ]:
# ── Haar Cascade (face detection) ─────────────────────────────────────────
# OpenCV ships cascades — find their path programmatically
import os
cascade_dir = os.path.join(cv2.data.haarcascades)
print("Cascade dir:", cascade_dir)

available = [f for f in os.listdir(cascade_dir) if f.endswith(".xml")]
print("\nAvailable cascades:")
for f in sorted(available): print(" ", f)


In [ ]:
# ── Haar face detector on a synthetic face-like image ─────────────────────
# (For a real face, just swap the image source)
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)
eye_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_eye.xml"
)

# Detect on a real image (swap path for your own photo)
# img = cv2.imread("photo.jpg")
# Synthetic demo only — real detection needs real faces
img_demo = np.full((200, 200, 3), 200, dtype=np.uint8)
gray_demo = cv2.cvtColor(img_demo, cv2.COLOR_BGR2GRAY)

faces = face_cascade.detectMultiScale(
    gray_demo,
    scaleFactor=1.1,    # how much image size is reduced at each scale
    minNeighbors=5,     # how many neighbours needed to retain detection
    minSize=(30, 30),
)
print(f"Faces detected: {len(faces)} (expected 0 on blank image)")
print("""
Usage on real image:
  faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(30,30))
  for (x, y, w, h) in faces:
      roi_gray = gray[y:y+h, x:x+w]
      cv2.rectangle(img, (x,y), (x+w,y+h), (255,0,0), 2)
      eyes = eye_cascade.detectMultiScale(roi_gray)
""")


---
## 12. Optical Flow & Background Subtraction <a id='12-video'></a>

In [ ]:
# ── Simulate a moving object across frames ─────────────────────────────────
def make_frame(t, h=300, w=400):
    """Circle moving across the frame as a function of time t."""
    frame = np.zeros((h, w, 3), dtype=np.uint8)
    cx = int(50 + (w - 100) * t)
    cy = 150 + int(40 * np.sin(t * 2 * np.pi))
    cv2.circle(frame, (cx, cy), 25, (0, 200, 255), -1)
    cv2.rectangle(frame, (120, 80), (220, 180), (100, 100, 100), -1)  # static BG object
    return frame

# Generate sequence
frames = [make_frame(t) for t in np.linspace(0, 1, 30)]

# ── Background subtraction (MOG2) ──────────────────────────────────────────
# Excellent for surveillance, motion detection
subtractor = cv2.createBackgroundSubtractorMOG2(
    history=50, varThreshold=16, detectShadows=True
)

fg_masks = []
for f in frames:
    fg = subtractor.apply(f)
    fg_masks.append(fg)

# Show first, mid, last frames and their masks
show_multi(
    ("Frame 0",  frames[0]),
    ("Frame 15", frames[15]),
    ("Frame 29", frames[29]),
    ("Mask 0",   fg_masks[0]),
    ("Mask 15",  fg_masks[15]),
    ("Mask 29",  fg_masks[29]),
    cols=3,
)
print("Shadow pixels = gray (127), foreground = white (255), background = black (0)")


In [ ]:
# ── Lucas-Kanade Optical Flow (sparse, tracks specific points) ─────────────
# Perfect for tracking corner points across frames

lk_params = dict(
    winSize=(21, 21),
    maxLevel=3,                          # pyramid levels
    criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01),
)

# Pick points to track in first frame
gray0 = cv2.cvtColor(frames[0], cv2.COLOR_BGR2GRAY)
p0 = cv2.goodFeaturesToTrack(gray0, maxCorners=20, qualityLevel=0.3, minDistance=7)

colors = np.random.randint(0, 255, (100, 3))
mask_track = np.zeros_like(frames[0])

prev_gray = gray0
tracked = frames[0].copy()

for frame in frames[1:]:
    cur_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    if p0 is None or len(p0) == 0:
        break
    
    # Calculate optical flow
    p1, status, err = cv2.calcOpticalFlowPyrLK(prev_gray, cur_gray, p0, None, **lk_params)
    
    good_new = p1[status == 1]
    good_old = p0[status == 1]
    
    for i, (new, old) in enumerate(zip(good_new, good_old)):
        a, b = new.ravel().astype(int)
        c, d = old.ravel().astype(int)
        cv2.line(mask_track, (a, b), (c, d), colors[i % len(colors)].tolist(), 2)
        cv2.circle(tracked, (a, b), 5, colors[i % len(colors)].tolist(), -1)
    
    tracked = cv2.add(tracked, mask_track)
    prev_gray = cur_gray
    p0 = good_new.reshape(-1, 1, 2)

show(tracked, "Lucas-Kanade sparse optical flow tracks", figsize=(8, 6))


In [ ]:
# ── Dense Optical Flow (Farneback) ────────────────────────────────────────
# Computes flow for EVERY pixel — useful for motion estimation, action recognition

prev_gray = cv2.cvtColor(frames[10], cv2.COLOR_BGR2GRAY)
curr_gray = cv2.cvtColor(frames[20], cv2.COLOR_BGR2GRAY)

flow = cv2.calcOpticalFlowFarneback(
    prev_gray, curr_gray,
    None,
    pyr_scale=0.5, levels=3, winsize=15,
    iterations=3, poly_n=5, poly_sigma=1.2,
    flags=0,
)

# Visualise as HSV: hue = direction, value = magnitude
magnitude, angle = cv2.cartToPolar(flow[..., 0], flow[..., 1])
hsv_flow = np.zeros((*prev_gray.shape, 3), dtype=np.uint8)
hsv_flow[..., 0] = angle * 180 / np.pi / 2   # Hue = direction [0,179]
hsv_flow[..., 1] = 255                         # Max saturation
hsv_flow[..., 2] = cv2.normalize(magnitude, None, 0, 255, cv2.NORM_MINMAX)
rgb_flow = cv2.cvtColor(hsv_flow, cv2.COLOR_HSV2BGR)

show_multi(
    ("Frame t=10",   frames[10]),
    ("Frame t=20",   frames[20]),
    ("Dense flow",   rgb_flow),
)
print("Hue = motion direction, Brightness = motion magnitude")


---
## 13. Camera Calibration & Homography <a id='13-calibration'></a>

In [ ]:
# ── Homography: perspective correction ────────────────────────────────────
# Classic use case: document scanner / bird's-eye view

src = make_test_image()
h, w = src.shape[:2]

# Simulate a "tilted document" by applying a perspective distortion first
src_pts = np.float32([[0,0],[w-1,0],[w-1,h-1],[0,h-1]])
dst_pts = np.float32([[80,60],[w-120,40],[w-80,h-60],[100,h-30]])

# Forward: flat → distorted
H_fwd, _ = cv2.findHomography(src_pts, dst_pts)
distorted = cv2.warpPerspective(src, H_fwd, (w, h))

# Inverse: distorted → corrected (simulate a scan)
H_inv, _ = cv2.findHomography(dst_pts, src_pts)
corrected = cv2.warpPerspective(distorted, H_inv, (w, h))

show_multi(
    ("Original (flat)",   src),
    ("Distorted (tilted)", distorted),
    ("Corrected (scanned)", corrected),
)


In [ ]:
# ── Camera calibration workflow ────────────────────────────────────────────
# In practice: collect 15-20 chessboard images from different angles
# Here we simulate the structure of the calibration pipeline

print("""
Camera Calibration Pipeline (run with real chessboard images):

1. Prepare object points (3D) for an NxM chessboard:
   objp = np.zeros((N*M, 3), np.float32)
   objp[:, :2] = np.mgrid[0:N, 0:M].T.reshape(-1, 2)

2. For each calibration image:
   gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
   ret, corners = cv2.findChessboardCorners(gray, (N, M), None)
   if ret:
       # Sub-pixel refinement
       corners2 = cv2.cornerSubPix(gray, corners, (11,11), (-1,-1), criteria)
       obj_points.append(objp)
       img_points.append(corners2)

3. Calibrate:
   ret, K, dist, rvecs, tvecs = cv2.calibrateCamera(
       obj_points, img_points, gray.shape[::-1], None, None
   )
   # K     = 3x3 camera intrinsic matrix [[fx,0,cx],[0,fy,cy],[0,0,1]]
   # dist  = distortion coefficients [k1,k2,p1,p2,k3]

4. Undistort new images:
   new_K, roi = cv2.getOptimalNewCameraMatrix(K, dist, (w,h), alpha=1)
   undistorted = cv2.undistort(img, K, dist, None, new_K)
   # Or use undistort maps for efficiency (pre-compute once):
   mapx, mapy = cv2.initUndistortRectifyMap(K, dist, None, new_K, (w,h), 5)
   undist = cv2.remap(img, mapx, mapy, cv2.INTER_LINEAR)
""")

# ── Simulate intrinsic matrix for a typical webcam ─────────────────────────
fx = fy = 800.0   # focal length in pixels
cx, cy = 320.0, 240.0
K = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]], dtype=np.float64)
dist = np.array([-0.3, 0.1, 0.001, 0.001, -0.05])   # typical distortion
print(f"Example camera matrix K:\n{K}")
print(f"Distortion coefficients: {dist}")


---
## 14. Integration with NumPy & PyTorch Tensors <a id='14-pytorch'></a>

In [ ]:
# ── OpenCV ↔ NumPy (zero-copy when possible) ──────────────────────────────
src = make_test_image()

# OpenCV images ARE NumPy arrays — no conversion needed
assert isinstance(src, np.ndarray)
print(f"Type: {type(src)}, dtype: {src.dtype}, shape: {src.shape}")

# Common dtype conversions
img_float32 = src.astype(np.float32) / 255.0    # normalise to [0, 1]
img_float64 = src.astype(np.float64)
img_back    = (img_float32 * 255).clip(0, 255).astype(np.uint8)

# ── OpenCV → PyTorch tensor ────────────────────────────────────────────────
# PyTorch expects: (C, H, W) and RGB order

def bgr_to_tensor(img_bgr, normalize=True):
    """Convert OpenCV BGR uint8 image to PyTorch-ready float tensor."""
    import importlib
    if importlib.util.find_spec("torch") is None:
        print("PyTorch not installed — showing equivalent NumPy ops")
        rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        chw = np.transpose(rgb, (2, 0, 1))          # HWC → CHW
        tensor_np = chw.astype(np.float32) / 255.0
        return tensor_np
    import torch
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)   # BGR → RGB
    chw = np.transpose(rgb, (2, 0, 1))               # HWC → CHW
    tensor = torch.from_numpy(chw.copy()).float()
    if normalize:
        tensor /= 255.0
    return tensor

t = bgr_to_tensor(src)
print(f"Tensor shape: {t.shape}  dtype: {t.dtype}  range: [{t.min():.3f}, {t.max():.3f}]")

def tensor_to_bgr(tensor):
    """Convert PyTorch tensor (C,H,W) float [0,1] back to OpenCV BGR uint8."""
    import importlib
    if importlib.util.find_spec("torch") is None:
        chw = tensor                                        # already numpy CHW
    else:
        import torch
        chw = tensor.detach().cpu().numpy()
    hwc_rgb = np.transpose(chw, (1, 2, 0))               # CHW → HWC
    hwc_rgb = (hwc_rgb * 255).clip(0, 255).astype(np.uint8)
    bgr = cv2.cvtColor(hwc_rgb, cv2.COLOR_RGB2BGR)
    return bgr

reconstructed = tensor_to_bgr(t)
show_multi(("Original", src), ("Tensor → BGR round-trip", reconstructed))
print("Max abs diff after round-trip:", np.abs(src.astype(int) - reconstructed.astype(int)).max())


In [ ]:
# ── ImageNet normalisation (standard for torchvision models) ─────────────
import numpy as np

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def preprocess_for_imagenet(img_bgr, size=(224, 224)):
    """
    Full preprocessing chain used by ResNet / EfficientNet / ViT etc.
    Returns: (1, 3, H, W) float32 numpy array  (ready for ort.run or torch forward)
    """
    # 1. Resize
    img = cv2.resize(img_bgr, size, interpolation=cv2.INTER_AREA)
    # 2. BGR → RGB, uint8 → float32 [0,1]
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    # 3. Normalise with ImageNet stats
    img = (img - IMAGENET_MEAN) / IMAGENET_STD
    # 4. HWC → CHW → NCHW
    img = np.transpose(img, (2, 0, 1))[np.newaxis]
    return img.astype(np.float32)

src = make_test_image()
batch = preprocess_for_imagenet(src)
print(f"Preprocessed batch shape: {batch.shape}")
print(f"Range: [{batch.min():.3f}, {batch.max():.3f}]")
print("Ready for: model.forward(torch.from_numpy(batch)) or onnxruntime session")


---
## 15. Real-World Pipeline: Preprocessing for CNN <a id='15-pipeline'></a>

In [ ]:
# ── Full inference-ready preprocessing pipeline ────────────────────────────
# Mirrors what a production CV system does before every forward pass.

class CVPreprocessor:
    """
    Configurable preprocessing pipeline connecting OpenCV to CNN inference.
    
    Supports: resize strategies, normalisation, augmentation, batch assembly.
    """
    
    def __init__(self, target_size=(640, 640), mode="letterbox",
                 mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)):
        self.target_size = target_size
        self.mode = mode
        self.mean = np.array(mean, dtype=np.float32)
        self.std  = np.array(std,  dtype=np.float32)
    
    def _letterbox(self, img):
        th, tw = self.target_size
        h, w = img.shape[:2]
        scale = min(th / h, tw / w)
        nh, nw = int(h * scale), int(w * scale)
        resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
        out = np.full((th, tw, 3), 114, dtype=np.uint8)
        y0, x0 = (th - nh) // 2, (tw - nw) // 2
        out[y0:y0+nh, x0:x0+nw] = resized
        return out, scale, (x0, y0)
    
    def _stretch(self, img):
        return cv2.resize(img, self.target_size[::-1],
                          interpolation=cv2.INTER_AREA), 1.0, (0, 0)
    
    def preprocess(self, img_bgr, augment=False):
        """BGR uint8 → CHW float32 normalised + metadata for post-processing."""
        # Optional augmentation (training time)
        if augment:
            img_bgr = self._random_augment(img_bgr)
        
        # Resize
        if self.mode == "letterbox":
            img, scale, offset = self._letterbox(img_bgr)
        else:
            img, scale, offset = self._stretch(img_bgr)
        
        # BGR → RGB, uint8 → float32 [0,1]
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        
        # Normalise
        img = (img - self.mean) / self.std
        
        # HWC → CHW
        img = np.transpose(img, (2, 0, 1))
        
        return img.astype(np.float32), {"scale": scale, "offset": offset,
                                         "orig_shape": img_bgr.shape[:2]}
    
    def _random_augment(self, img):
        # Flip
        if np.random.rand() > 0.5:
            img = cv2.flip(img, 1)
        # HSV jitter
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
        hsv[..., 1] *= np.random.uniform(0.8, 1.2)   # saturation
        hsv[..., 2] *= np.random.uniform(0.8, 1.2)   # value
        hsv = np.clip(hsv, 0, 255).astype(np.uint8)
        return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
    
    def batch(self, images, augment=False):
        """Process a list of BGR images into a (N, C, H, W) float32 array."""
        tensors, metas = zip(*[self.preprocess(img, augment) for img in images])
        return np.stack(tensors), list(metas)


# ── Test the pipeline ──────────────────────────────────────────────────────
prep = CVPreprocessor(target_size=(640, 640), mode="letterbox")

imgs = [make_test_image(), make_test_image(300, 500), make_test_image(600, 400)]
batch_arr, metas = prep.batch(imgs)

print(f"Batch shape : {batch_arr.shape}")
print(f"dtype       : {batch_arr.dtype}")
print(f"value range : [{batch_arr.min():.3f}, {batch_arr.max():.3f}]")
print("\nPer-image metadata:")
for i, m in enumerate(metas):
    print(f"  img[{i}]: orig_shape={m['orig_shape']}, scale={m['scale']:.3f}, offset={m['offset']}")


In [ ]:
# ── Post-processing: map detections back to original coordinates ──────────
# After a detector outputs bboxes in the resized/letterboxed space,
# you need to map them back to the original image.

def map_boxes_back(boxes_xyxy, meta):
    """
    boxes_xyxy: (N, 4) array in [x1, y1, x2, y2] format (letterboxed space)
    meta: dict returned by CVPreprocessor.preprocess()
    Returns: boxes in original image coordinates
    """
    ox, oy = meta["offset"]
    scale  = meta["scale"]
    
    boxes = boxes_xyxy.copy().astype(np.float32)
    boxes[:, [0, 2]] -= ox    # remove x padding
    boxes[:, [1, 3]] -= oy    # remove y padding
    boxes /= scale            # unscale
    
    # Clip to original image boundaries
    oh, ow = meta["orig_shape"]
    boxes[:, [0, 2]] = boxes[:, [0, 2]].clip(0, ow)
    boxes[:, [1, 3]] = boxes[:, [1, 3]].clip(0, oh)
    return boxes

# ── NMS from scratch using cv2.dnn.NMSBoxes ────────────────────────────────
def apply_nms(boxes_xywh, scores, iou_threshold=0.45, score_threshold=0.25):
    """
    boxes_xywh: list of [x, y, w, h]  (cv2 format)
    scores    : list of confidence scores
    Returns   : kept indices
    """
    if len(boxes_xywh) == 0:
        return []
    indices = cv2.dnn.NMSBoxes(boxes_xywh, scores, score_threshold, iou_threshold)
    return indices.flatten() if len(indices) > 0 else []

# Demo
dummy_boxes  = [[50, 50, 100, 100], [55, 55, 100, 100], [300, 300, 80, 80]]
dummy_scores = [0.9, 0.85, 0.7]
kept = apply_nms(dummy_boxes, dummy_scores, iou_threshold=0.5)
print(f"NMS kept indices: {kept}")
print(f"Boxes after NMS: {[dummy_boxes[i] for i in kept]}")


In [ ]:
# ── Video I/O reference (works identically with webcam or file) ──────────
print("""
Video capture & write template:
================================================

cap = cv2.VideoCapture(0)              # 0 = webcam, or "video.mp4"
cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

fps = cap.get(cv2.CAP_PROP_FPS)
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter("output.mp4", fourcc, fps, (1280, 720))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    
    # ── your processing here ──
    processed = frame  # e.g. run your model
    # ─────────────────────────
    
    out.write(processed)

    # In a real script (not Jupyter):
    cv2.imshow("Frame", processed)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
out.release()
cv2.destroyAllWindows()
================================================
""")


---
## 🎓 Summary & What's Next

### What you've covered

| Section | Key APIs |
|---------|----------|
| I/O & colors | `imread`, `imwrite`, `cvtColor`, `split/merge`, `inRange` |
| Drawing & ROI | `line/rect/circle/ellipse`, `putText`, NumPy slicing |
| Geometric | `resize`, `warpAffine`, `warpPerspective`, `flip` |
| Filtering | `GaussianBlur`, `medianBlur`, `bilateralFilter`, `filter2D`, CLAHE |
| Morphology | `erode/dilate`, `morphologyEx` |
| Edges | `Sobel`, `Canny`, `HoughLinesP`, `HoughCircles` |
| Contours | `findContours`, `contourArea`, `moments`, `approxPolyDP` |
| Histograms | `calcHist`, `equalizeHist`, CLAHE, `calcBackProject` |
| Features | ORB, SIFT, BFMatcher, FLANN, Homography |
| Detection | HOG+SVM, Haar cascades |
| Video | MOG2, Lucas-Kanade, Farneback dense flow |
| Calibration | `findHomography`, `calibrateCamera`, `undistort` |
| PyTorch bridge | BGR→tensor, letterbox, ImageNet normalisation, NMS |

### Recommended next steps

1. **DNN module** — `cv2.dnn.readNet()` loads ONNX/TF/Caffe models; pairs naturally with your PyTorch exports.
2. **ONNX Runtime** — `ort.InferenceSession` + this notebook's preprocessing pipeline = production-ready inference.
3. **OpenCV CUDA** — `cv2.cuda.*` mirrors the CPU API with GPU acceleration.
4. **ArUco markers** — `cv2.aruco` for pose estimation and AR.
5. **Stereo vision** — `cv2.StereoBM`, `StereoSGBM` for depth maps from stereo pairs.
